#LLM re-ranker to a FAISS-based RAG pipeline

You’ll build:

FAISS retriever (candidate pool)

Gemini re-ranker (cross-encoder–style scoring via prompting)

Final answer generator with inline citations

0) Install dependencies

In [1]:
!pip install google-generativeai faiss-cpu sentence-transformers numpy pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

#1) End-to-end Reranker for RAG

In [3]:
"""
RAG with FAISS + Gemini Re-Ranker (1.5 Flash)
---------------------------------------------
What this script does:
1) Builds embeddings for a small corpus using Sentence-Transformers
2) Indexes them in FAISS (exact, cosine via inner product on unit vectors)
3) Retrieves a candidate pool with FAISS (fast)
4) Re-ranks the pool using Gemini 1.5 Flash as a learned re-ranker
5) Generates a grounded answer with citations

Notes:
- Re-ranking uses a strict JSON scoring prompt so we can parse numeric scores.
- Keep your candidate pool modest (e.g., 10–50) to control cost/latency.
"""

# ===================== Imports =====================
import os
import re
import uuid
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

# ===================== Config ======================
# 1) Set your Gemini key
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
assert GOOGLE_API_KEY and GOOGLE_API_KEY != "YOUR_GEMINI_API_KEY", "Please set GOOGLE_API_KEY env var or edit the placeholder."
genai.configure(api_key=GOOGLE_API_KEY)

# 2) Models
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEMINI_MODEL = "gemini-1.5-flash"  # or "gemini-1.5-flash-latest"

# 3) RAG hyperparams
CANDIDATE_POOL_K = 12    # how many docs to retrieve from FAISS before re-ranking
FINAL_TOP_K = 5          # how many to keep after re-ranking

RUN_ID = str(uuid.uuid4())[:8]

# ===================== Toy Corpus ===================
# Feel free to replace with your own docs (docs, FAQs, notes, etc.)
DOCS = [
    "FAISS is a library for efficient similarity search over dense vectors.",
    "RAG combines retrieval with generation to ground LLM outputs in external knowledge.",
    "Gemini 1.5 Flash is a fast multimodal model suitable for low-latency applications.",
    "To build a RAG pipeline you need an embedder, a vector index, a retriever, and a generator.",
    "Cosine similarity measures the angle between two vectors; unit normalization helps.",
    "IVF and PQ in FAISS enable approximate search and compression for large-scale corpora.",
    "Re-ranking rescored candidates using a cross-encoder or LLM to improve precision.",
    "Maximal Marginal Relevance (MMR) balances relevance and diversity in retrieval.",
    "Sentence-Transformers provides easy text embedding models for semantic search.",
    "Citations in answers improve transparency and make it easier to verify claims.",
    "You can combine FAISS retrieval with Gemini generation for grounded responses.",
    "RAG systems should show sources and avoid hallucinations with careful prompts.",
]

# ===================== Embeddings ===================
print("Loading embedding model and encoding corpus...")
embedder = SentenceTransformer(EMBED_MODEL)

def embed_texts(texts: List[str]) -> np.ndarray:
    """Returns L2-normalized float32 embeddings for cosine/IP search."""
    vecs = embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return vecs.astype("float32")

doc_vecs = embed_texts(DOCS)
dim = doc_vecs.shape[1]
print(f"Embedded {len(DOCS)} docs, dimension={dim}")

# ===================== FAISS Index ==================
# Exact search with Inner Product (on unit vectors -> cosine similarity)
index = faiss.IndexFlatIP(dim)
index.add(doc_vecs)
id2text = {i: DOCS[i] for i in range(len(DOCS))}
print("FAISS index built. ntotal =", index.ntotal)

# ===================== FAISS Retrieval ==============
def retrieve_candidates(query: str, pool_k: int = CANDIDATE_POOL_K) -> List[Dict]:
    """Top-k FAISS retrieval (fast, approximate relevance)."""
    qv = embed_texts([query])
    D, I = index.search(qv, pool_k)
    out = []
    for s, idx in zip(D[0], I[0]):
        out.append({"doc_id": int(idx), "faiss_score": float(s), "text": id2text[int(idx)]})
    return out

# ===================== Gemini Re-Ranker =============
# We'll prompt Gemini to produce a JSON score for each (query, passage) pair.
RERANK_SYSTEM_INSTRUCTION = (
    "You are a reranking model. Score how relevant a passage is to a user query.\n"
    "Only output a compact JSON object: {\"score\": <0..100>} with no other text.\n"
    "Scoring rubric:\n"
    "- 90-100: Directly answers key aspects of the query with high precision\n"
    "- 70-89: Mostly relevant; helpful context but not exact\n"
    "- 40-69: Somewhat related; partial or generic\n"
    "- 1-39: Marginally related or off-topic\n"
    "- 0: Not related\n"
)

def build_rerank_prompt(query: str, passage: str) -> str:
    return (
        f"Query: {query}\n\n"
        f"Passage: {passage}\n\n"
        "Return JSON with a single numeric key 'score' in 0..100. Nothing else."
    )

def parse_score(text: str) -> float:
    """
    Extract a number 0..100 from Gemini output. We ask for JSON but we parse defensively.
    Returns a float in [0, 100] or 0.0 if parsing fails.
    """
    # Try to capture "score": <number>
    m = re.search(r'"score"\s*:\s*([0-9]+(?:\.[0-9]+)?)', text)
    if m:
        val = float(m.group(1))
        return max(0.0, min(100.0, val))
    # Fallback: first number we see
    m2 = re.search(r'([0-9]+(?:\.[0-9]+)?)', text)
    if m2:
        val = float(m2.group(1))
        return max(0.0, min(100.0, val))
    return 0.0

def rerank_with_gemini(query: str, candidates: List[Dict],
                       model_name: str = GEMINI_MODEL,
                       show_progress: bool = True) -> List[Dict]:
    """
    Calls Gemini once per candidate to get a relevance score (0..100).
    Sorts candidates by Gemini score (desc). Returns list with 'llm_score'.
    """
    model = genai.GenerativeModel(model_name, system_instruction=RERANK_SYSTEM_INSTRUCTION)
    scored = []
    it = tqdm(candidates, desc="Re-ranking with Gemini") if show_progress else candidates

    for c in it:
        prompt = build_rerank_prompt(query, c["text"])
        try:
            resp = model.generate_content(prompt)
            score = parse_score(resp.text or "")
        except Exception as e:
            print("Gemini error (scored 0):", e)
            score = 0.0
        out = dict(c)
        out["llm_score"] = score
        scored.append(out)

    scored.sort(key=lambda x: x["llm_score"], reverse=True)
    return scored

# ===================== Answer Generation =============
GEN_SYSTEM_INSTRUCTION = (
    "You answer using ONLY the provided passages. Synthesize them faithfully, and cite doc_ids like [id]. "
    "If information is missing, say so clearly. Be concise, neutral, and avoid hallucinations."
)

def generate_answer(query: str, contexts: List[Dict],
                    model_name: str = GEMINI_MODEL,
                    system_instruction: str = GEN_SYSTEM_INSTRUCTION) -> str:
    """Generate a grounded answer with inline citations."""
    ctx_block = "\n".join([f"[{c['doc_id']}] {c['text']}" for c in contexts])
    prompt = (
        f"User question: {query}\n\n"
        f"Passages:\n{ctx_block}\n\n"
        "Write a short, accurate answer grounded in the passages. Add citations [id] inline."
    )
    model = genai.GenerativeModel(model_name, system_instruction=system_instruction)
    resp = model.generate_content(prompt)
    return resp.text

# ===================== Demo Run ======================
if __name__ == "__main__":
    query = "How do I build a RAG pipeline with FAISS and Gemini, and why might I add a re-ranker?"

    print("\n=== 1) FAISS Candidate Retrieval ===")
    pool = retrieve_candidates(query, pool_k=CANDIDATE_POOL_K)
    df_pool = pd.DataFrame(pool)
    print(df_pool[["doc_id", "faiss_score", "text"]].to_string(index=False))

    print("\n=== 2) Gemini Re-Ranking (cross-encoder style) ===")
    reranked = rerank_with_gemini(query, pool, model_name=GEMINI_MODEL, show_progress=True)
    df_re = pd.DataFrame(reranked)
    print(df_re[["doc_id", "faiss_score", "llm_score", "text"]].to_string(index=False))

    print(f"\nTaking top-{FINAL_TOP_K} after re-ranking...")
    top_ctx = reranked[:FINAL_TOP_K]

    print("\n=== 3) Final Answer (grounded, with citations) ===")
    answer = generate_answer(query, top_ctx, model_name=GEMINI_MODEL)
    print(answer)

    # (Optional) Save debug CSVs
    df_pool.assign(run_id=RUN_ID, stage="faiss").to_csv(f"retrieval_{RUN_ID}.csv", index=False)
    df_re.assign(run_id=RUN_ID, stage="reranked").to_csv(f"reranked_{RUN_ID}.csv", index=False)
    print(f"\nSaved: retrieval_{RUN_ID}.csv and reranked_{RUN_ID}.csv")

Loading embedding model and encoding corpus...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedded 12 docs, dimension=384
FAISS index built. ntotal = 12

=== 1) FAISS Candidate Retrieval ===
 doc_id  faiss_score                                                                                        text
      3     0.654500 To build a RAG pipeline you need an embedder, a vector index, a retriever, and a generator.
     10     0.414275              You can combine FAISS retrieval with Gemini generation for grounded responses.
      1     0.293913         RAG combines retrieval with generation to ground LLM outputs in external knowledge.
      6     0.288146           Re-ranking rescored candidates using a cross-encoder or LLM to improve precision.
      2     0.199650          Gemini 1.5 Flash is a fast multimodal model suitable for low-latency applications.
     11     0.185841              RAG systems should show sources and avoid hallucinations with careful prompts.
      5     0.170855      IVF and PQ in FAISS enable approximate search and compression for large-scale corp

Re-ranking with Gemini: 100%|██████████| 12/12 [00:13<00:00,  1.14s/it]


 doc_id  faiss_score  llm_score                                                                                        text
      3     0.654500       40.0 To build a RAG pipeline you need an embedder, a vector index, a retriever, and a generator.
     10     0.414275       40.0              You can combine FAISS retrieval with Gemini generation for grounded responses.
      1     0.293913       40.0         RAG combines retrieval with generation to ground LLM outputs in external knowledge.
      6     0.288146       40.0           Re-ranking rescored candidates using a cross-encoder or LLM to improve precision.
     11     0.185841       40.0              RAG systems should show sources and avoid hallucinations with careful prompts.
      5     0.170855       40.0      IVF and PQ in FAISS enable approximate search and compression for large-scale corpora.
      7     0.156866       40.0             Maximal Marginal Relevance (MMR) balances relevance and diversity in retrieval.
      0 

#How it works (quickly)

FAISS gives you a fast top-K candidate pool by vector similarity.

Gemini re-ranker looks at each (query, passage) pair and assigns a relevance score (0–100).

You sort by the LLM score and keep the top few to feed the generator.

The final Gemini generation step answers with inline citations to the selected passages.